In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

In [0]:
df = spark.read.format("parquet")\
          .load("abfss://bronze@adlsolympicsproject.dfs.core.windows.net/athletes")

In [0]:
display(df)



In [0]:
# Atheletes is our master data where we are hoing to perform so many pyspark transformations

#We are replacing null values from birth place and birth country

df = df.fillna({"birth_place": "xyz", "birth_country": "abc", "residence_place": "unknown", "residence_country": "pqr" })

In [0]:
display(df)

In [0]:
df_filtered = df.filter((col("current")== "True") & (col("name").isin('GALSTYAN Slavik','HARUTYUNYAN Arsen', 'TEVANYAN Vazgen' )))
df_filtered.display()

In [0]:
#transforming the data type of height and weight

df = df.withColumn('height', col("height").cast(FloatType()))\
       .withColumn('weight', col("weight").cast(FloatType()))
display(df)


In [0]:
#descending order of height and asc of weight

df_sorted = df.sort('height', 'weight', ascending=[0,1]).filter(col('weight')>0)
display(df_sorted)

In [0]:
#replacing United Stats with US

df_sorted = df_sorted.withColumn('nationality', regexp_replace('nationality', 'United States', 'US'))
display(df_sorted)

In [0]:
df.groupBy('code').agg(count('code').alias('total_count')).display()

In [0]:
df_sorted = df_sorted.withColumnRenamed("code", "athelete_id")
display(df_sorted)

In [0]:
df_sorted = df_sorted.withColumn('occupation', split('occupation', ','))
display(df_sorted)

#now occupation is turned as list rather tham string

In [0]:
df_sorted.columns

In [0]:
df_final = df_sorted.select('athelete_id',
 'current',
 'name',
 'name_short',
 'name_tv',
 'gender',
 'function',
 'country_code',
 'country',
 'country_long',
 'nationality',
 'nationality_long',
 'nationality_code',
 'height',
 'weight')

In [0]:
display(df_final)

Databricks visualization. Run in Databricks to view.

In [0]:
#Window function - cumulative sum based on nationality
#Here with rowsBetween function - it will take all values for past, current and futute rows else it will consider only past and current
#You can do same in sql too

df_final.withColumn('cumWeight',sum('weight').over(Window.partitionBy('nationality').orderBy("height").rowsBetween(Window.unboundedPreceding,Window.unboundedFollowing))).display()

In [0]:
df_final.write.format("delta")\
        .mode("append")\
        .option("path", "abfss://silver@adlsolympicsproject.dfs.core.windows.net/atheletes")\
        .saveAsTable("olympics.silver.atheletes")